# 2.2. Créer une V3 du notebook avec un algorithme tree-based SynapseML
Ce notebook remplace l'approche scikit-learn par un entraînement Spark distribué avec `LightGBMRegressor` de SynapseML.

## Convention commune
Cette version enregistre ses runs dans `ds_prediction_velos_v3_synapseml` et conserve le modèle logique `prediction_velos_horaires`.

In [ ]:
import mlflow
import mlflow.spark
from pyspark.sql import functions as F
from pyspark.ml import Pipeline
from pyspark.ml.feature import StringIndexer, VectorAssembler
from pyspark.ml.evaluation import RegressionEvaluator
from synapse.ml.lightgbm import LightGBMRegressor

In [ ]:
source_path = "Files/comptage-velo-donnees-compteurs.csv"
df = spark.read.option("header", True).option("sep", ";").csv(source_path)
train_base = (
    df
    .withColumn("date_time", F.to_timestamp("Date et heure de comptage"))
    .withColumn("jour", F.to_date("date_time"))
    .withColumn("heure", F.hour("date_time"))
    .withColumn("jour_semaine", F.dayofweek("jour"))
    .withColumn("mois", F.month("jour"))
    .withColumn("station", F.col("Nom du site de comptage"))
    .withColumn("nb_velos", F.col("Comptage horaire").cast("double"))
    .select("station", "jour_semaine", "mois", "heure", "nb_velos")
    .dropna()
)
display(train_base.limit(10))
train_df, test_df = train_base.randomSplit([0.8, 0.2], seed=42)

In [ ]:
station_indexer = StringIndexer(inputCol="station", outputCol="station_index", handleInvalid="keep")
assembler = VectorAssembler(inputCols=["station_index", "jour_semaine", "mois", "heure"], outputCol="features")
regressor = LightGBMRegressor(labelCol="nb_velos", featuresCol="features", objective="regression", numIterations=100, learningRate=0.1, numLeaves=31)
pipeline = Pipeline(stages=[station_indexer, assembler, regressor])
model = pipeline.fit(train_df)
predictions = model.transform(test_df)
display(predictions.select("station", "heure", "nb_velos", "prediction"))

## Entraîner le modèle SynapseML
Cette étape reste entièrement en Spark et entraîne un `LightGBMRegressor` distribué.

In [ ]:
rmse = RegressionEvaluator(labelCol="nb_velos", predictionCol="prediction", metricName="rmse").evaluate(predictions)
mae = RegressionEvaluator(labelCol="nb_velos", predictionCol="prediction", metricName="mae").evaluate(predictions)
r2 = RegressionEvaluator(labelCol="nb_velos", predictionCol="prediction", metricName="r2").evaluate(predictions)
print({"rmse": rmse, "mae": mae, "r2": r2})

In [ ]:
experiment_name = "ds_prediction_velos_v3_synapseml"
run_name = "v3_synapseml_lightgbm"
registered_model_name = "prediction_velos_horaires"

mlflow.set_experiment(experiment_name)
with mlflow.start_run(run_name=run_name):
    mlflow.log_param("registered_model_name", registered_model_name)
    mlflow.log_param("model_type", "LightGBMRegressor")
    mlflow.log_metric("rmse", rmse)
    mlflow.log_metric("mae", mae)
    mlflow.log_metric("r2", r2)
    mlflow.spark.log_model(model, artifact_path="model")

## À retenir
La V3 garde les données dans Spark et exploite SynapseML pour un entraînement distribué, plus adapté à un passage à l'échelle.
## Exercice
Tester d'autres paramètres `LightGBMRegressor`, puis comparer les métriques de la V3 avec celles de la V1 et de la V2.